# PRACH interference detection — campagne d'expériences (révision Springer Nature)

**Avant de lancer :** menu *Exécution → Modifier le type d'exécution → GPU (T4)*.

Ce notebook :
1. écrit les 4 modules Python (simulateur PRACH format 0, caractéristiques, expériences, analyse) ;
2. fait un **test rapide** (2–3 min) ;
3. lance la **campagne complète** : 10 runs × 7 détecteurs (corrélation, XGBoost, CNN, Conv1D, LSTM, Transformer, Conv1D-IQ) ;
4. calcule les statistiques, les tableaux LaTeX et les figures, puis crée `results.zip` à télécharger.

Les résultats sont écrits **sur Google Drive** après chaque modèle : si Colab se déconnecte, relancez toutes les cellules dans l'ordre (le test rapide peut être sauté) : la campagne reprend là où elle s'était arrêtée.


In [ ]:
# 1. GPU + Google Drive
import tensorflow as tf, os
print('TensorFlow', tf.__version__, '| GPU :', tf.config.list_physical_devices('GPU'))
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/prach_revision/results'
os.makedirs(OUT, exist_ok=True)
print('Résultats dans :', OUT)

In [ ]:
# 2. Dépendances
!pip -q install xgboost psutil
import xgboost; print('xgboost', xgboost.__version__)

In [ ]:
%%writefile prach_sim.py
"""
prach_sim.py -- 3GPP-style 5G NR PRACH Format 0 simulator for interference detection.

Signal chain (one sample = one PRACH occasion observed at the gNB):
  desired UE preamble  (own-cell root u0, random cyclic shift v_d, timing advance TA_d)
  + optional interferer (class 1: intra-cell = same root u0, other cyclic shift;
                         class 2: inter-cell = neighbour-cell root, random delay)
  each through an independent multipath fading channel (ETU or EPA delay profile,
  Jakes Doppler via sum-of-sinusoids, N_RX receive antennas), with its own CFO,
  + AWGN at the requested SNR (defined on the PRACH subcarriers).
Receiver: CP removal, unitary FFT, extraction of the 839 PRACH subcarriers,
frequency-domain correlation with candidate root sequences (power delay profiles).

Numerology (TS 38.211, Format 0): L_RA = 839, Delta f_RA = 1.25 kHz,
N_u = 24576 kappa*Tc (0.8 ms), N_CP = 3168 kappa*Tc (103.1 us).
The simulator works at fs = 1.28 MHz (N_FFT = 1024 = 24576/24, N_CP = 132 = 3168/24),
which is sufficient because the PRACH occupies 839 x 1.25 kHz = 1.05 MHz.
"""
import numpy as np

# ----------------------------------------------------------------- constants
L_RA = 839
SCS = 1.25e3
N_FFT = 1024
FS = N_FFT * SCS                 # 1.28 MHz
N_CP = 132                       # 3168 samples @ 30.72 MHz -> 132 @ 1.28 MHz
N_CS = 26                        # zeroCorrelationZoneConfig = 5 (unrestricted set)
N_PRE_PER_ROOT = L_RA // N_CS    # 32 preambles per root
PAD = 96                         # zero padding (75 us) for delayed arrivals

# 3GPP TS 36.104 Annex B delay profiles (delays in ns, relative powers in dB)
DELAY_PROFILES = {
    "ETU": ([0, 50, 120, 200, 230, 500, 1600, 2300, 5000],
            [-1.0, -1.0, -1.0, 0.0, 0.0, 0.0, -3.0, -5.0, -7.0]),
    "EPA": ([0, 30, 70, 90, 110, 190, 410],
            [0.0, -1.0, -2.0, -3.0, -8.0, -17.2, -20.8]),
}

DEFAULT_CFG = dict(
    u0=129,                          # own-cell physical root index
    neighbour_roots=(300, 501, 688), # roots of the neighbouring cells (known to the gNB)
    unseen_roots=(77, 412, 777),     # roots never seen in training (robustness test)
    profile="ETU", doppler_hz=70.0, n_rx=2, n_sin=16,
    cfo_desired_hz=270.0, cfo_interf_max_hz=270.0,
    ta_max_us=14.0,                  # desired and intra-cell UEs (inside the ZCZ)
    inter_delay_max_us=40.0,         # inter-cell interferer: unsynchronised arrival
)


def zc_root(u):
    n = np.arange(L_RA)
    return np.exp(-1j * np.pi * u * n * (n + 1) / L_RA)


def preamble_freq(u, v):
    """Frequency-domain preamble y_{u,v}(k) per TS 38.211 6.3.3.1 (unit average power)."""
    x = zc_root(u)
    x_uv = np.roll(x, -v * N_CS)                # x_u((n + C_v) mod L_RA), C_v = v*N_CS
    return np.fft.fft(x_uv) / np.sqrt(L_RA)


# PRACH subcarriers centred in the FFT grid
_SC = (np.arange(L_RA) - L_RA // 2) % N_FFT


def preamble_time(u, v):
    """Time-domain PRACH symbol with CP at fs = 1.28 MHz, zero padded."""
    grid = np.zeros(N_FFT, complex)
    grid[_SC] = preamble_freq(u, v)
    s = np.fft.ifft(grid) * np.sqrt(N_FFT)      # unitary -> unit power per used subcarrier
    s = np.concatenate([s[-N_CP:], s])
    return np.concatenate([s, np.zeros(PAD, complex)])


N_TOT = N_CP + N_FFT + PAD
_t = np.arange(N_TOT) / FS
_f = np.fft.fftfreq(N_TOT, 1 / FS)
_tc = np.linspace(0, _t[-1], 17)
_W_INTERP = np.zeros((len(_tc), N_TOT))
for _j in range(len(_tc)):
    _e = np.zeros(len(_tc)); _e[_j] = 1.0
    _W_INTERP[_j] = np.interp(_t, _tc, _e)


def _fading_channel(rng, B, profile, fd, n_rx, n_sin):
    """Time-varying tap gains, shape (B, n_rx, n_taps, N_TOT); total average power 1."""
    delays_ns, p_db = DELAY_PROFILES[profile]
    p = 10 ** (np.asarray(p_db) / 10)
    p = p / p.sum()
    L = len(p)
    alpha = rng.uniform(0, 2 * np.pi, (B, n_rx, L, n_sin, 1))
    phi = rng.uniform(0, 2 * np.pi, (B, n_rx, L, n_sin, 1))
    # sum-of-sinusoids evaluated on a coarse grid (the burst lasts ~1 ms << 1/fd) and
    # linearly interpolated to the sample grid
    gc = np.exp(1j * (2 * np.pi * fd * np.cos(alpha) * _tc + phi)).sum(axis=3) / np.sqrt(n_sin)
    g = gc @ _W_INTERP
    g *= np.sqrt(p)[None, None, :, None]
    return g, np.asarray(delays_ns) * 1e-9


def _propagate(rng, s, B, delay_s, cfo_hz, power, cfg):
    """Pass B copies of the time signal s through independent channels.
    delay_s, cfo_hz, power: arrays of shape (B,). Returns (B, n_rx, N_TOT)."""
    g, tap_d = _fading_channel(rng, B, cfg["profile"], cfg["doppler_hz"], cfg["n_rx"], cfg["n_sin"])
    S = np.fft.fft(s)
    tot_d = delay_s[:, None] + tap_d[None, :]                   # (B, L)
    ramps = np.exp(-2j * np.pi * _f[None, None, :] * tot_d[:, :, None])
    delayed = np.fft.ifft(S[None, None, :] * ramps, axis=-1)     # (B, L, N_TOT) fractional delays
    y = np.einsum("brln,bln->brn", g, delayed)
    y *= np.exp(2j * np.pi * cfo_hz[:, None, None] * _t[None, None, :])
    return y * np.sqrt(power)[:, None, None]


def receiver_freq(y):
    """CP removal + unitary FFT + PRACH subcarrier extraction. y: (B, n_rx, N_TOT) -> (B, n_rx, 839)."""
    Y = np.fft.fft(y[..., N_CP:N_CP + N_FFT], axis=-1) / np.sqrt(N_FFT)
    return Y[..., _SC]


_ROOT_FREQ = {}


def _root_conj(u):
    if u not in _ROOT_FREQ:
        _ROOT_FREQ[u] = np.conj(preamble_freq(u, 0))
    return _ROOT_FREQ[u]


def correlation_maps(Yf, roots):
    """Power delay profiles P_u[tau] = sum_a |IDFT(Y_a[k] conj(X_u[k]))|^2,
    returns (B, len(roots), 839)."""
    out = []
    for u in roots:
        r = np.fft.ifft(Yf * _root_conj(u)[None, None, :], axis=-1) * np.sqrt(L_RA)
        out.append((np.abs(r) ** 2).sum(axis=1))
    return np.stack(out, axis=1)


def generate(rng, n_per_cell, snr_db_list, isr_db_list, cfg=None, inter_roots=None, batch=128):
    """Balanced dataset: for each (class, SNR, ISR) cell, n_per_cell independent occasions.
    Class 0 (no interference) is generated for every ISR value too, so that classes are balanced
    and every (SNR, ISR) bin contains the three classes (ISR is irrelevant for class 0).
    Returns dict with Yf (B,n_rx,839) complex64, labels, snr, isr, metadata."""
    cfg = dict(DEFAULT_CFG, **(cfg or {}))
    inter_roots = tuple(inter_roots or cfg["neighbour_roots"])
    jobs = [(c, s, i) for c in (0, 1, 2) for s in snr_db_list for i in isr_db_list]
    Yf_all, lab, snr, isr, meta = [], [], [], [], []
    s_cache = {}

    def st(u, v):
        if (u, v) not in s_cache:
            s_cache[(u, v)] = preamble_time(u, v)
        return s_cache[(u, v)]

    for (c, s_db, i_db) in jobs:
        done = 0
        while done < n_per_cell:
            B = min(batch, n_per_cell - done)
            v_d = rng.integers(0, N_PRE_PER_ROOT, B)
            ta_d = rng.uniform(0, cfg["ta_max_us"], B) * 1e-6
            y = np.zeros((B, cfg["n_rx"], N_TOT), complex)
            # desired UEs: group by preamble index to reuse the base waveform
            for v in np.unique(v_d):
                m = v_d == v
                y[m] += _propagate(rng, st(cfg["u0"], v), m.sum(), ta_d[m],
                                   np.full(m.sum(), cfg["cfo_desired_hz"]), np.ones(m.sum()), cfg)
            u_i = np.full(B, -1)
            v_i = np.full(B, -1)
            if c > 0:
                if c == 1:   # intra-cell: same root, different cyclic shift, synchronised (TA in ZCZ)
                    u_i[:] = cfg["u0"]
                    v_i = (v_d + rng.integers(1, N_PRE_PER_ROOT, B)) % N_PRE_PER_ROOT
                    d_i = rng.uniform(0, cfg["ta_max_us"], B) * 1e-6
                else:        # inter-cell: neighbour root, any preamble, unsynchronised delay
                    u_i = rng.choice(inter_roots, B)
                    v_i = rng.integers(0, N_PRE_PER_ROOT, B)
                    d_i = rng.uniform(0, cfg["inter_delay_max_us"], B) * 1e-6
                cfo_i = rng.uniform(-cfg["cfo_interf_max_hz"], cfg["cfo_interf_max_hz"], B)
                p_i = np.full(B, 10 ** (i_db / 10))
                for key in set(zip(u_i, v_i)):
                    m = (u_i == key[0]) & (v_i == key[1])
                    y[m] += _propagate(rng, st(*key), m.sum(), d_i[m], cfo_i[m], p_i[m], cfg)
            # AWGN: unit-power preamble per subcarrier -> noise variance 1/SNR per sample
            sigma2 = 10 ** (-s_db / 10)
            y += np.sqrt(sigma2 / 2) * (rng.standard_normal(y.shape) + 1j * rng.standard_normal(y.shape))
            Yf_all.append(receiver_freq(y).astype(np.complex64))
            lab += [c] * B
            snr += [s_db] * B
            isr += [i_db if c > 0 else np.nan] * B
            meta.append(np.c_[v_d, ta_d * 1e6, u_i, v_i])
            done += B
    return dict(Yf=np.concatenate(Yf_all), y=np.array(lab), snr=np.array(snr, float),
                isr=np.array(isr, float), isr_bin=np.repeat([i for (_, _, i) in jobs], n_per_cell).astype(float),
                meta=np.concatenate(meta), cfg=cfg)


In [ ]:
%%writefile features.py
"""Input representations, engineered features and the conventional correlation-based detector."""
import numpy as np
import prach_sim as ps

# detection window of preamble v in the own-root delay profile: (-C_v + [0, N_CS)) mod L_RA
_WIN = np.stack([(np.arange(ps.N_CS) - ps.N_CS * v) % ps.L_RA for v in range(ps.N_PRE_PER_ROOT)])


def candidate_roots(cfg):
    return (cfg["u0"],) + tuple(cfg["neighbour_roots"])


def maps_from(Yf, cfg):
    """Correlation maps (B, R, 839) normalised by each sample's own-root median (noise-floor proxy)."""
    P = ps.correlation_maps(Yf, candidate_roots(cfg))
    floor = np.median(P[:, 0, :], axis=1)[:, None, None] + 1e-12
    return (P / floor).astype(np.float32)


def deep_inputs(Yf, cfg, frame=8):
    """Returns the three tensors used by the deep models:
       cmap  (B, R, 839, 1)  log-compressed correlation map  -> 2D CNN
       iq    (B, 839, 2*n_rx) per-sample standardised freq-domain IQ  -> Conv1D
       tok   (B, T, R*frame)  framed correlation map tokens  -> LSTM / Transformer"""
    M = np.log1p(maps_from(Yf, cfg))
    B, R, L = M.shape
    cmap = M[..., None]
    iq = np.concatenate([Yf.real, Yf.imag], axis=1).transpose(0, 2, 1)
    iq = (iq - iq.mean(axis=(1, 2), keepdims=True)) / (iq.std(axis=(1, 2), keepdims=True) + 1e-9)
    Lp = int(np.ceil(L / frame) * frame)
    Mp = np.pad(M, ((0, 0), (0, 0), (0, Lp - L)))
    tok = Mp.reshape(B, R, Lp // frame, frame).transpose(0, 2, 1, 3).reshape(B, Lp // frame, R * frame)
    return cmap.astype(np.float32), iq.astype(np.float32), tok.astype(np.float32)


def engineered_features(Yf, cfg):
    """Tabular features for XGBoost (names in FEATURE_NAMES)."""
    P = maps_from(Yf, cfg)                         # (B, R, L), own-root median = 1
    own = P[:, 0, :]
    wpk = np.sort(own[:, _WIN].max(axis=2), axis=1)[:, ::-1]   # sorted per-window peaks (B, 32)
    f = {}
    f["own_peak1"] = np.log1p(wpk[:, 0])
    f["own_peak2"] = np.log1p(wpk[:, 1])
    f["own_peak3"] = np.log1p(wpk[:, 2])
    f["own_peak2_over_peak1"] = wpk[:, 1] / (wpk[:, 0] + 1e-9)
    f["own_mean_over_median"] = own.mean(axis=1)
    f["own_p95_over_median"] = np.quantile(own, 0.95, axis=1)
    f["own_n_windows_gt8"] = (wpk > 8).sum(axis=1)
    f["own_n_windows_gt16"] = (wpk > 16).sum(axis=1)
    nb = P[:, 1:, :]
    nb_peak = nb.max(axis=2)
    nb_psr = nb_peak / (np.median(nb, axis=2) + 1e-9)
    f["nb_peak_max"] = np.log1p(nb_peak.max(axis=1))
    f["nb_psr_max"] = np.log1p(nb_psr.max(axis=1))
    f["nb_peak_over_own_peak"] = nb_peak.max(axis=1) / (wpk[:, 0] + 1e-9)
    f["nb_mean_over_own_median"] = nb.mean(axis=(1, 2))
    rx_pow = (np.abs(Yf) ** 2).mean(axis=(1, 2))
    raw_own = ps.correlation_maps(Yf, (cfg["u0"],))[:, 0, :]
    f["energy_over_own_floor"] = np.log(rx_pow / (np.median(raw_own, axis=1) / ps.L_RA * 1 + 1e-12))
    f["own_kurtosis"] = ((own - own.mean(1, keepdims=True)) ** 4).mean(1) / (own.var(1) ** 2 + 1e-12)
    return np.stack([f[k] for k in FEATURE_NAMES], axis=1).astype(np.float32)


FEATURE_NAMES = ["own_peak1", "own_peak2", "own_peak3", "own_peak2_over_peak1", "own_mean_over_median",
                 "own_p95_over_median", "own_n_windows_gt8", "own_n_windows_gt16", "nb_peak_max",
                 "nb_psr_max", "nb_peak_over_own_peak", "nb_mean_over_own_median",
                 "energy_over_own_floor", "own_kurtosis"]


# ------------------------------------------------------------ conventional detector
def baseline_stats(Yf, cfg):
    """Classical test statistics computed from matched-filter outputs (no learning):
       r_floor = second-largest own-root window peak / own-root noise floor
       r_main  = second-largest own-root window peak / largest own-root window peak
       t_inter = largest neighbour-root peak / neighbour-root noise floor"""
    P = ps.correlation_maps(Yf, candidate_roots(cfg))
    own = P[:, 0, :]
    wpk = np.sort(own[:, _WIN].max(axis=2), axis=1)
    r_floor = wpk[:, -2] / (np.median(own, axis=1) + 1e-12)
    r_main = wpk[:, -2] / (wpk[:, -1] + 1e-12)
    nb = P[:, 1:, :]
    t_inter = (nb.max(axis=2) / (np.median(nb, axis=2) + 1e-12)).max(axis=1)
    return r_floor, r_main, t_inter


def baseline_predict(stats, th):
    r_floor, r_main, t_inter = stats
    pred = np.zeros(len(r_floor), int)
    pred[(r_floor > th[0]) & (r_main > th[1])] = 1
    pred[t_inter > th[2]] = 2          # a neighbour-root peak takes precedence
    return pred


def baseline_fit(stats, y, n_grid=20):
    """Grid search of the three detection thresholds maximising macro-F1 on the validation set
    (the usual way thresholds of a correlation detector are calibrated)."""
    from sklearn.metrics import f1_score
    grids = [np.quantile(s, np.linspace(0.05, 0.995, n_grid)) for s in stats]
    best, bt = -1, None
    for a in grids[0]:
        for b in grids[1]:
            for c in grids[2]:
                sc = f1_score(y, baseline_predict(stats, (a, b, c)), average="macro")
                if sc > best:
                    best, bt = sc, (a, b, c)
    return bt


In [ ]:
%%writefile experiments.py
"""
experiments.py -- multi-run benchmark of PRACH interference detectors.

For every run r = 1..N_RUNS a fresh, independent dataset is generated (disjoint random
streams for train / validation / test / robustness sets) and every detector is retrained
with its own seed. Results are appended to CSV files after each (run, model) so that an
interrupted Colab session can be resumed.
"""
import os, json, time, platform
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix)
import prach_sim as ps
import features as ft

CLASSES = ["none", "intra", "inter"]
MODELS = ["Correlation", "XGBoost", "CNN", "Conv1D", "LSTM", "Transformer", "Conv1D-IQ"]

DEFAULT_EXP = dict(
    n_runs=10,
    snr_db=[-18, -15, -12, -9, -6, -3, 0, 5, 10],
    isr_db=[-9, -6, -3, 0, 3],
    n_train_cell=150, n_val_cell=40, n_test_cell=60, n_rob_cell=30,
    epochs=60, batch=64, lr=1e-3, patience=8,
    models=MODELS,
    out_dir="results",
)


# =============================================================== deep models (Keras)
def _keras():
    import tensorflow as tf
    from tensorflow.keras import layers as L, Model, Input
    return tf, L, Model, Input


def build_cnn(shape):
    tf, L, Model, Input = _keras()
    x0 = Input(shape)                                   # (R=4 roots, 839 delays, 1)
    x = x0
    for f, k in [(16, (2, 7)), (32, (2, 5)), (64, (2, 3))]:
        x = L.Conv2D(f, k, padding="same")(x)
        x = L.BatchNormalization()(x)
        x = L.ReLU()(x)
        x = L.MaxPooling2D((1, 2))(x)
    x = L.GlobalMaxPooling2D()(x)
    x = L.Dense(64, activation="relu")(x)
    x = L.Dropout(0.3)(x)
    return Model(x0, L.Dense(3, activation="softmax")(x), name="CNN")


def build_conv1d(shape, name="Conv1D"):
    tf, L, Model, Input = _keras()
    x0 = Input(shape)              # (839 delays, R roots) or, for the ablation, (839 subcarriers, 2*N_RX)
    x = x0
    for f, k, d in [(32, 9, 1), (64, 7, 1), (64, 5, 2), (64, 5, 4)]:
        x = L.Conv1D(f, k, padding="same", dilation_rate=d)(x)
        x = L.BatchNormalization()(x)
        x = L.ReLU()(x)
        if d == 1:
            x = L.MaxPooling1D(2)(x)
    x = L.Concatenate()([L.GlobalAveragePooling1D()(x), L.GlobalMaxPooling1D()(x)])
    x = L.Dense(64, activation="relu")(x)
    x = L.Dropout(0.3)(x)
    return Model(x0, L.Dense(3, activation="softmax")(x), name=name)


def build_lstm(shape):
    tf, L, Model, Input = _keras()
    x0 = Input(shape)                                   # (T tokens, R*frame)
    x = L.TimeDistributed(L.Dense(64, activation="relu"))(x0)   # token embedding
    x = L.Bidirectional(L.LSTM(64, return_sequences=True))(x)
    x = L.Bidirectional(L.LSTM(64, return_sequences=True))(x)
    x = L.Concatenate()([L.GlobalMaxPooling1D()(x), L.GlobalAveragePooling1D()(x)])  # temporal pooling
    x = L.Dense(64, activation="relu")(x)
    x = L.Dropout(0.3)(x)
    return Model(x0, L.Dense(3, activation="softmax")(x), name="LSTM")


def build_transformer(shape, d_model=64, heads=4, ff=128, blocks=2, drop=0.1):
    """Pre-LayerNorm Transformer encoder (more stable to train than the post-LN variant)."""
    tf, L, Model, Input = _keras()
    T = shape[0]
    pos = np.arange(T)[:, None]
    i = np.arange(d_model)[None, :]
    ang = pos / np.power(10000, (2 * (i // 2)) / d_model)
    pe = np.where(i % 2 == 0, np.sin(ang), np.cos(ang)).astype("float32")[None]
    x0 = Input(shape)
    x = L.Dense(d_model)(L.LayerNormalization()(x0)) + tf.constant(pe)
    for _ in range(blocks):
        h = L.LayerNormalization()(x)
        x = x + L.Dropout(drop)(L.MultiHeadAttention(heads, d_model // heads, dropout=drop)(h, h))
        h = L.LayerNormalization()(x)
        x = x + L.Dropout(drop)(L.Dense(d_model)(L.Dense(ff, activation="relu")(h)))
    x = L.LayerNormalization()(x)
    x = L.Concatenate()([L.GlobalMaxPooling1D()(x), L.GlobalAveragePooling1D()(x)])
    x = L.Dense(64, activation="relu")(x)
    x = L.Dropout(0.3)(x)
    return Model(x0, L.Dense(3, activation="softmax")(x), name="Transformer")


BUILDERS = {"CNN": (build_cnn, "cmap"), "Conv1D": (build_conv1d, "seq"),
            "Conv1D-IQ": (lambda sh: build_conv1d(sh, "Conv1D_IQ"), "iq"),
            "LSTM": (build_lstm, "tok"), "Transformer": (build_transformer, "tok")}


# =============================================================== helpers
def make_split(seed, n_cell, exp, inter_roots=None, cfg=None):
    return ps.generate(np.random.default_rng(seed), n_cell, exp["snr_db"], exp["isr_db"],
                       cfg=cfg, inter_roots=inter_roots)


def prepare(d, cfg):
    t = time.perf_counter()
    cmap, iq, tok = ft.deep_inputs(d["Yf"], cfg)
    t_deep = (time.perf_counter() - t) / len(d["y"])
    t = time.perf_counter()
    F = ft.engineered_features(d["Yf"], cfg)
    t_feat = (time.perf_counter() - t) / len(d["y"])
    seq = cmap[..., 0].transpose(0, 2, 1).copy()      # (B, 839, R): roots as channels
    return dict(cmap=cmap, seq=seq, iq=iq, tok=tok, feat=F, t_deep=t_deep, t_feat=t_feat)


def metrics(y, p):
    m = dict(accuracy=accuracy_score(y, p),
             macro_f1=f1_score(y, p, average="macro"),
             weighted_f1=f1_score(y, p, average="weighted"),
             macro_precision=precision_score(y, p, average="macro", zero_division=0),
             macro_recall=recall_score(y, p, average="macro", zero_division=0))
    pc = precision_score(y, p, average=None, labels=[0, 1, 2], zero_division=0)
    rc = recall_score(y, p, average=None, labels=[0, 1, 2], zero_division=0)
    fc = f1_score(y, p, average=None, labels=[0, 1, 2], zero_division=0)
    for k, c in enumerate(CLASSES):
        m[f"precision_{c}"], m[f"recall_{c}"], m[f"f1_{c}"] = pc[k], rc[k], fc[k]
    yb, pb = (y > 0).astype(int), (p > 0).astype(int)          # interference present vs absent
    m.update(bin_precision=precision_score(yb, pb, zero_division=0),
             bin_recall=recall_score(yb, pb, zero_division=0), bin_f1=f1_score(yb, pb, zero_division=0))
    return m


def per_condition(run, model, tag, d, p):
    rows = []
    for s in np.unique(d["snr"]):
        for i in np.unique(d["isr_bin"]):
            m = (d["snr"] == s) & (d["isr_bin"] == i)
            y, q = d["y"][m], p[m]
            rows.append(dict(run=run, model=model, test_set=tag, snr_db=s, isr_db=i, n=int(m.sum()),
                             n_correct=int((y == q).sum()),
                             **{f"n_{c}": int((y == k).sum()) for k, c in enumerate(CLASSES)},
                             **{f"correct_{c}": int(((y == k) & (q == k)).sum()) for k, c in enumerate(CLASSES)},
                             **{f"pred_{c}": int((q == k).sum()) for k, c in enumerate(CLASSES)}))
    return rows


def _append(path, rows):
    df = pd.DataFrame(rows)
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def hardware():
    info = dict(platform=platform.platform(), processor=platform.processor(), python=platform.python_version())
    try:
        import tensorflow as tf
        info["tensorflow"] = tf.__version__
        info["gpus"] = [g.name for g in tf.config.list_physical_devices("GPU")]
        try:
            info["gpu_details"] = [tf.config.experimental.get_device_details(g) for g in tf.config.list_physical_devices("GPU")]
        except Exception:
            pass
    except Exception:
        pass
    try:
        import xgboost
        info["xgboost"] = xgboost.__version__
    except Exception:
        pass
    try:
        import psutil
        info["ram_gb"] = round(psutil.virtual_memory().total / 2**30, 1)
        info["cpu_count"] = psutil.cpu_count()
    except Exception:
        info["cpu_count"] = os.cpu_count()
    return info


# =============================================================== one run
def run_once(r, exp, done, log=print):
    out = exp["out_dir"]
    cfg = dict(ps.DEFAULT_CFG)
    base = 1000 * r
    t0 = time.perf_counter()
    tr = make_split(base + 1, exp["n_train_cell"], exp)
    va = make_split(base + 2, exp["n_val_cell"], exp)
    te = make_split(base + 3, exp["n_test_cell"], exp)
    rob_root = make_split(base + 4, exp["n_rob_cell"], exp, inter_roots=cfg["unseen_roots"])
    rob_epa = make_split(base + 5, exp["n_rob_cell"], exp, cfg=dict(profile="EPA", doppler_hz=5.0))
    log(f"[run {r}] data generated in {time.perf_counter()-t0:.0f}s "
        f"(train {len(tr['y'])}, val {len(va['y'])}, test {len(te['y'])})")
    if r == 1:
        rows = []
        for name, d in [("train", tr), ("validation", va), ("test", te),
                        ("test_unseen_roots", rob_root), ("test_EPA5", rob_epa)]:
            g = pd.DataFrame(dict(cls=[CLASSES[k] for k in d["y"]], snr=d["snr"], isr=d["isr_bin"]))
            for (c, s, i), n in g.groupby(["cls", "snr", "isr"]).size().items():
                rows.append(dict(split=name, cls=c, snr_db=s, isr_db=i, n=n))
        pd.DataFrame(rows).to_csv(os.path.join(out, "dataset_summary.csv"), index=False)
    X = {k: prepare(d, cfg) for k, d in [("tr", tr), ("va", va), ("te", te), ("rr", rob_root), ("re", rob_epa)]}
    sets = {"test": te, "test_unseen_roots": rob_root, "test_EPA5": rob_epa}
    keymap = {"test": "te", "test_unseen_roots": "rr", "test_EPA5": "re"}

    for name in exp["models"]:
        if (r, name) in done:
            continue
        log(f"[run {r}] {name} ...")
        extra = dict(thresholds="")
        if name == "Correlation":
            stats = {k: ft.baseline_stats(d["Yf"], cfg) for k, d in [("va", va)] + [(keymap[s], sets[s]) for s in sets]}
            t = time.perf_counter()
            th = ft.baseline_fit(stats["va"], va["y"])
            fit_s = time.perf_counter() - t
            pred = {}
            for s in sets:
                t = time.perf_counter()
                pred[s] = ft.baseline_predict(stats[keymap[s]], th)
                if s == "test":
                    inf = (time.perf_counter() - t) / len(te["y"])
            n_params, epochs, pre = 3, np.nan, X["te"]["t_deep"]
            extra = dict(thresholds=json.dumps([float(v) for v in th]))
        elif name == "XGBoost":
            import xgboost as xgb
            m = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, subsample=0.8,
                                  colsample_bytree=0.8, reg_lambda=1.0, gamma=0.1, tree_method="hist",
                                  objective="multi:softprob", eval_metric="mlogloss",
                                  early_stopping_rounds=20, random_state=r, n_jobs=-1)
            t = time.perf_counter()
            m.fit(X["tr"]["feat"], tr["y"], eval_set=[(X["va"]["feat"], va["y"])], verbose=False)
            fit_s = time.perf_counter() - t
            pred = {}
            for s in sets:
                t = time.perf_counter()
                pred[s] = m.predict(X[keymap[s]]["feat"])
                if s == "test":
                    inf = (time.perf_counter() - t) / len(te["y"])
            epochs = int(m.best_iteration) + 1
            n_params = int(sum(len(tdf) for _, tdf in m.get_booster().trees_to_dataframe().groupby("Tree")))
            pre = X["te"]["t_feat"]
            if r == 1:
                imp = pd.DataFrame(dict(feature=ft.FEATURE_NAMES, gain=m.feature_importances_))
                imp.to_csv(os.path.join(out, "xgb_feature_importance.csv"), index=False)
        else:
            import tensorflow as tf
            tf.keras.utils.set_random_seed(r)
            builder, key = BUILDERS[name]
            model = builder(X["tr"][key].shape[1:])
            model.compile(tf.keras.optimizers.Adam(exp["lr"], clipnorm=1.0),
                          loss="sparse_categorical_crossentropy", metrics=["accuracy"])
            if r == 1:
                with open(os.path.join(out, "model_summaries.txt"), "a") as fh:
                    model.summary(print_fn=lambda s, **k: fh.write(s + "\n"))
                    fh.write(f"Trainable parameters: {int(sum(np.prod(v.shape) for v in model.trainable_weights))}\n\n")
            cb = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=exp["patience"],
                                                   restore_best_weights=True)]
            t = time.perf_counter()
            h = model.fit(X["tr"][key], tr["y"], validation_data=(X["va"][key], va["y"]),
                          epochs=exp["epochs"], batch_size=exp["batch"], callbacks=cb, verbose=0)
            fit_s = time.perf_counter() - t
            if r == 1:
                pd.DataFrame(h.history).assign(epoch=lambda d: np.arange(1, len(d) + 1)).to_csv(
                    os.path.join(out, f"history_{name}.csv"), index=False)
            pred = {}
            model.predict(X["te"][key][:256], batch_size=256, verbose=0)     # warm-up
            for s in sets:
                t = time.perf_counter()
                pred[s] = model.predict(X[keymap[s]][key], batch_size=256, verbose=0).argmax(1)
                if s == "test":
                    inf = (time.perf_counter() - t) / len(te["y"])
            epochs = len(h.history["loss"])
            n_params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
            pre = X["te"]["t_deep"]
            tf.keras.backend.clear_session()

        row = dict(run=r, model=name, fit_s=fit_s, infer_ms_per_sample=1e3 * inf,
                   preproc_ms_per_sample=1e3 * pre, n_params=n_params, epochs=epochs,
                   **metrics(te["y"], pred["test"]), **extra)
        for s in ["test_unseen_roots", "test_EPA5"]:
            mm = metrics(sets[s]["y"], pred[s])
            row[f"{s}_accuracy"], row[f"{s}_macro_f1"] = mm["accuracy"], mm["macro_f1"]
        _append(os.path.join(out, "runs.csv"), [row])
        pc = []
        for s in sets:
            pc += per_condition(r, name, s, sets[s], pred[s])
        _append(os.path.join(out, "per_condition.csv"), pc)
        cm = confusion_matrix(te["y"], pred["test"], labels=[0, 1, 2])
        _append(os.path.join(out, "confusion.csv"),
                [dict(run=r, model=name, true=CLASSES[i], pred=CLASSES[j], count=int(cm[i, j]))
                 for i in range(3) for j in range(3)])
        log(f"[run {r}] {name}: acc={row['accuracy']:.4f} macroF1={row['macro_f1']:.4f} "
            f"fit={fit_s:.1f}s inf={row['infer_ms_per_sample']:.3f}ms/sample")


def run_all(exp=None, log=print):
    exp = dict(DEFAULT_EXP, **(exp or {}))
    os.makedirs(exp["out_dir"], exist_ok=True)
    with open(os.path.join(exp["out_dir"], "config.json"), "w") as fh:
        json.dump(dict(experiment=exp, simulator=ps.DEFAULT_CFG, hardware=hardware(),
                       numerology=dict(L_RA=ps.L_RA, SCS=ps.SCS, N_FFT=ps.N_FFT, FS=ps.FS, N_CP=ps.N_CP,
                                       N_CS=ps.N_CS)), fh, indent=2, default=str)
    path = os.path.join(exp["out_dir"], "runs.csv")
    done = set()
    if os.path.exists(path):
        d = pd.read_csv(path)
        done = set(zip(d["run"], d["model"]))
    for r in range(1, exp["n_runs"] + 1):
        if all((r, m) in done for m in exp["models"]):
            continue
        run_once(r, exp, done, log)


In [ ]:
%%writefile analysis.py
"""analysis.py -- statistics, LaTeX tables and figures from the CSV files written by experiments.py."""
import os, json, itertools
import numpy as np
import pandas as pd
from scipy import stats as st

ORDER = ["Correlation", "XGBoost", "CNN", "Conv1D", "LSTM", "Transformer", "Conv1D-IQ"]
# validated categorical palette (fixed order); the classical baseline is a neutral gray
COLORS = {"XGBoost": "#2a78d6", "CNN": "#eb6834", "Conv1D": "#1baf7a", "LSTM": "#eda100",
          "Transformer": "#e87ba4", "Conv1D-IQ": "#008300", "Correlation": "#6f6e69"}
MARKERS = {"XGBoost": "o", "CNN": "s", "Conv1D": "^", "LSTM": "D", "Transformer": "v", "Conv1D-IQ": "P", "Correlation": "x"}


def ci95(x):
    x = np.asarray(x, float)
    if len(x) < 2:
        return np.nan
    return st.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))


def holm(pvals):
    p = np.asarray(pvals, float)
    o = np.argsort(p)
    adj = np.empty_like(p)
    run = 0.0
    for k, i in enumerate(o):
        run = max(run, (len(p) - k) * p[i])
        adj[i] = min(1.0, run)
    return adj


def load_runs(out="results"):
    """Read runs.csv robustly: rows written without the baseline-only 'thresholds' column
    (all learning models) have one field fewer than the header, so the field is re-inserted."""
    import csv
    with open(os.path.join(out, "runs.csv"), newline="") as fh:
        rows = list(csv.reader(fh))
    head, body = rows[0], rows[1:]
    k = head.index("thresholds") if "thresholds" in head else None
    fixed = []
    for r in body:
        if k is not None and len(r) == len(head) - 1:
            r = r[:k] + [""] + r[k:]
        if len(r) != len(head):
            raise ValueError(f"unexpected row length {len(r)}: {r[:3]}")
        fixed.append(r)
    d = pd.DataFrame(fixed, columns=head)
    for c in d.columns:
        if c not in ("model", "thresholds"):
            d[c] = pd.to_numeric(d[c], errors="coerce")
    return d


def summarise(out="results"):
    d = load_runs(out)
    models = [m for m in ORDER if m in d["model"].unique()]
    cols = ["accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall",
            "f1_none", "f1_intra", "f1_inter", "bin_precision", "bin_recall", "bin_f1",
            "test_unseen_roots_accuracy", "test_unseen_roots_macro_f1", "test_EPA5_accuracy", "test_EPA5_macro_f1",
            "fit_s", "infer_ms_per_sample", "preproc_ms_per_sample", "n_params", "epochs"]
    rows = []
    for m in models:
        g = d[d.model == m]
        r = dict(model=m, n_runs=len(g))
        for c in cols:
            r[c + "_mean"], r[c + "_std"], r[c + "_ci95"] = g[c].mean(), g[c].std(ddof=1), ci95(g[c])
        rows.append(r)
    S = pd.DataFrame(rows)
    S.to_csv(os.path.join(out, "summary.csv"), index=False)

    # ---- significance: Friedman over runs + pairwise Wilcoxon signed-rank (Holm-corrected)
    res = {}
    for metric in ["macro_f1", "accuracy"]:
        W = d.pivot(index="run", columns="model", values=metric)[models].dropna()
        out_m = {"n_runs": int(len(W))}
        if len(W) >= 3 and len(models) >= 3:
            fr = st.friedmanchisquare(*[W[m] for m in models])
            out_m["friedman_chi2"], out_m["friedman_p"] = float(fr.statistic), float(fr.pvalue)
            out_m["mean_rank"] = {m: float(v) for m, v in W.rank(axis=1, ascending=False).mean().items()}
        pairs, pv, diffs = [], [], []
        for a, b in itertools.combinations(models, 2):
            x, y = W[a].values, W[b].values
            if np.allclose(x, y):
                p = 1.0
            else:
                p = float(st.wilcoxon(x, y).pvalue) if len(W) >= 5 else np.nan
            pairs.append((a, b)); pv.append(p); diffs.append(float(np.mean(x - y)))
        adj = holm(np.nan_to_num(pv, nan=1.0))
        out_m["pairwise"] = [dict(a=a, b=b, mean_diff=df_, p=p, p_holm=float(q))
                             for (a, b), df_, p, q in zip(pairs, diffs, pv, adj)]
        res[metric] = out_m
    with open(os.path.join(out, "stats.json"), "w") as fh:
        json.dump(res, fh, indent=2)
    pd.DataFrame(res["macro_f1"]["pairwise"]).to_csv(os.path.join(out, "pairwise_macro_f1.csv"), index=False)
    return S, res


def _pm(S, m, c, scale=100, nd=2):
    r = S[S.model == m].iloc[0]
    return f"{scale*r[c+'_mean']:.{nd}f} $\\pm$ {scale*r[c+'_std']:.{nd}f}"


def latex_tables(out="results"):
    S = pd.read_csv(os.path.join(out, "summary.csv"))
    models = [m for m in ORDER if m in S.model.values]
    L = []
    L.append("% Table: detection performance (mean +- std over runs, %)")
    L.append("\\begin{tabular}{lcccccc}\n\\toprule")
    L.append("Model & Accuracy & Macro-F1 & Weighted-F1 & F1 (none) & F1 (intra) & F1 (inter) \\\\\n\\midrule")
    for m in models:
        L.append(f"{m} & " + " & ".join(_pm(S, m, c) for c in
                 ["accuracy", "macro_f1", "weighted_f1", "f1_none", "f1_intra", "f1_inter"]) + " \\\\")
    L.append("\\bottomrule\n\\end{tabular}\n")
    L.append("% Table: robustness (macro-F1 %, mean +- std)")
    L.append("\\begin{tabular}{lccc}\n\\toprule\nModel & Nominal test & Unseen inter-cell roots & EPA-5\\,Hz channel \\\\\n\\midrule")
    for m in models:
        L.append(f"{m} & " + " & ".join(_pm(S, m, c) for c in
                 ["macro_f1", "test_unseen_roots_macro_f1", "test_EPA5_macro_f1"]) + " \\\\")
    L.append("\\bottomrule\n\\end{tabular}\n")
    L.append("% Table: computational cost (mean +- std)")
    L.append("\\begin{tabular}{lcccc}\n\\toprule\nModel & Trainable params & Training time [s] & Inference [ms/sample] & Pre-processing [ms/sample] \\\\\n\\midrule")
    for m in models:
        r = S[S.model == m].iloc[0]
        L.append(f"{m} & {r['n_params_mean']:,.0f} & {r['fit_s_mean']:.1f} $\\pm$ {r['fit_s_std']:.1f} & "
                 f"{r['infer_ms_per_sample_mean']:.3f} $\\pm$ {r['infer_ms_per_sample_std']:.3f} & "
                 f"{r['preproc_ms_per_sample_mean']:.3f} \\\\")
    L.append("\\bottomrule\n\\end{tabular}\n")
    open(os.path.join(out, "tables.tex"), "w").write("\n".join(L))


# ------------------------------------------------------------------ figures
def _style():
    import matplotlib as mpl
    mpl.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                         "axes.grid": True, "grid.color": "#e4e3dd", "grid.linewidth": 0.6,
                         "axes.edgecolor": "#8a8980", "axes.labelcolor": "#2b2b29",
                         "xtick.color": "#55544f", "ytick.color": "#55544f", "legend.frameon": False,
                         "lines.linewidth": 2.0, "lines.markersize": 5, "pdf.fonttype": 42,
                         "savefig.bbox": "tight"})


def _curve(pc, by, models, ax, metric="accuracy"):
    for m in models:
        g = pc[(pc.model == m)]
        per_run = g.groupby(["run", by])[["n", "n_correct"]].sum().reset_index()
        per_run["acc"] = per_run.n_correct / per_run.n
        agg = per_run.groupby(by)["acc"].agg(["mean", lambda x: ci95(x)])
        agg.columns = ["mean", "ci"]
        x = agg.index.values
        ax.plot(x, 100 * agg["mean"], marker=MARKERS[m], color=COLORS[m], label=m,
                linestyle="--" if m == "Correlation" else "-")
        lo = np.clip(100 * (agg["mean"] - agg["ci"]), 0, 100)
        hi = np.clip(100 * (agg["mean"] + agg["ci"]), 0, 100)
        ax.fill_between(x, lo, hi, color=COLORS[m], alpha=0.12, linewidth=0)
    ax.set_ylim(0, 100)


def figures(out="results"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _style()
    fd = os.path.join(out, "figures")
    os.makedirs(fd, exist_ok=True)
    d = load_runs(out)
    pc = pd.read_csv(os.path.join(out, "per_condition.csv"))
    models = [m for m in ORDER if m in d.model.unique()]
    pct = pc[pc.test_set == "test"]

    # 1. accuracy vs SNR
    fig, ax = plt.subplots(figsize=(4.8, 3.4))
    _curve(pct, "snr_db", models, ax)
    ax.set_xlabel("SNR [dB]"); ax.set_ylabel("Accuracy [%] (mean, 95% CI)")
    ax.legend(ncol=4, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.2))
    fig.savefig(os.path.join(fd, "acc_vs_snr.pdf")); plt.close(fig)

    # 2. accuracy vs ISR (interference classes only, all SNRs)
    fig, ax = plt.subplots(figsize=(4.8, 3.4))
    _curve(pct, "isr_db", models, ax)
    ax.set_xlabel("ISR bin [dB] (all classes)"); ax.set_ylabel("Accuracy [%] (mean, 95% CI)")
    ax.legend(ncol=4, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.2))
    fig.savefig(os.path.join(fd, "acc_vs_isr.pdf")); plt.close(fig)

    # 3. per-class recall vs SNR (one panel per class)
    fig, axs = plt.subplots(1, 3, figsize=(9.6, 3.0), sharey=True)
    for k, c in enumerate(["none", "intra", "inter"]):
        for m in models:
            g = pct[pct.model == m].groupby(["run", "snr_db"])[[f"n_{c}", f"correct_{c}"]].sum().reset_index()
            g["r"] = g[f"correct_{c}"] / g[f"n_{c}"]
            a = g.groupby("snr_db")["r"].mean()
            axs[k].plot(a.index, 100 * a.values, marker=MARKERS[m], color=COLORS[m], label=m,
                        linestyle="--" if m == "Correlation" else "-")
        axs[k].set_title({"none": "No interference", "intra": "Intra-cell", "inter": "Inter-cell"}[c])
        axs[k].set_xlabel("SNR [dB]")
    axs[0].set_ylabel("Recall [%]")
    axs[-1].legend(fontsize=7, ncol=1)
    fig.savefig(os.path.join(fd, "recall_per_class_vs_snr.pdf")); plt.close(fig)

    # 4. confusion matrices (row-normalised, summed over runs)
    cm = pd.read_csv(os.path.join(out, "confusion.csv"))
    labs = ["none", "intra", "inter"]
    n = len(models)
    fig, axs = plt.subplots(1, n, figsize=(2.0 * n, 2.3))
    for ax, m in zip(np.atleast_1d(axs), models):
        M = cm[cm.model == m].pivot_table(index="true", columns="pred", values="count", aggfunc="sum").reindex(index=labs, columns=labs).values
        Mn = M / M.sum(axis=1, keepdims=True)
        ax.imshow(Mn, cmap="Blues", vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{100*Mn[i,j]:.1f}", ha="center", va="center", fontsize=7,
                        color="white" if Mn[i, j] > 0.6 else "#2b2b29")
        ax.set_xticks(range(3)); ax.set_xticklabels(labs, fontsize=7)
        ax.set_yticks(range(3)); ax.set_yticklabels(labs if ax is np.atleast_1d(axs)[0] else [], fontsize=7)
        ax.set_title(m, fontsize=8); ax.grid(False)
    np.atleast_1d(axs)[0].set_ylabel("True class")
    fig.supxlabel("Predicted class", fontsize=8)
    fig.savefig(os.path.join(fd, "confusion_matrices.pdf")); plt.close(fig)

    # 5. macro-F1 with 95% CI
    S = pd.read_csv(os.path.join(out, "summary.csv")).set_index("model").loc[models]
    fig, ax = plt.subplots(figsize=(4.8, 3.0))
    x = np.arange(len(models))
    ax.bar(x, 100 * S["macro_f1_mean"], yerr=100 * S["macro_f1_ci95"], color=[COLORS[m] for m in models],
           width=0.6, capsize=3, edgecolor="white", linewidth=2)
    for xi, v in zip(x, S["macro_f1_mean"]):
        ax.text(xi, 100 * v + 1, f"{100*v:.1f}", ha="center", fontsize=7, color="#2b2b29")
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=20, fontsize=8)
    ax.set_ylabel("Macro-F1 [%] (mean, 95% CI)")
    ax.set_ylim(max(0, 100 * S["macro_f1_mean"].min() - 15), 100)
    ax.grid(axis="x", visible=False)
    fig.savefig(os.path.join(fd, "macro_f1_ci.pdf")); plt.close(fig)

    # 6. computational cost (two separate charts, log scale)
    for col, lab, fn in [("fit_s", "Training time [s]", "fit_time.pdf"),
                         ("infer_ms_per_sample", "Inference latency [ms/sample]", "inference_time.pdf")]:
        fig, ax = plt.subplots(figsize=(4.8, 2.8))
        ax.bar(x, S[col + "_mean"], yerr=S[col + "_std"], color=[COLORS[m] for m in models], width=0.6,
               capsize=3, edgecolor="white", linewidth=2)
        ax.set_yscale("log"); ax.set_ylabel(lab)
        ax.set_xticks(x); ax.set_xticklabels(models, rotation=20, fontsize=8)
        ax.grid(axis="x", visible=False)
        fig.savefig(os.path.join(fd, fn)); plt.close(fig)

    # 7. learning curves of run 1
    hs = [m for m in models if os.path.exists(os.path.join(out, f"history_{m}.csv"))]
    if hs:
        fig, axs = plt.subplots(1, 2, figsize=(8.4, 3.0))
        for m in hs:
            h = pd.read_csv(os.path.join(out, f"history_{m}.csv"))
            axs[0].plot(h.epoch, h.loss, color=COLORS[m], linestyle=":", linewidth=1.5)
            axs[0].plot(h.epoch, h.val_loss, color=COLORS[m], label=m)
            axs[1].plot(h.epoch, 100 * h.accuracy, color=COLORS[m], linestyle=":", linewidth=1.5)
            axs[1].plot(h.epoch, 100 * h.val_accuracy, color=COLORS[m], label=m)
        axs[0].set_ylabel("Loss"); axs[1].set_ylabel("Accuracy [%]")
        for a in axs:
            a.set_xlabel("Epoch")
        axs[1].legend(fontsize=7, title="solid: validation, dotted: training", title_fontsize=7)
        fig.savefig(os.path.join(fd, "learning_curves.pdf")); plt.close(fig)


def full_report(out="results"):
    S, res = summarise(out)
    latex_tables(out)
    figures(out)
    return S, res


In [ ]:
# 3. Test rapide (petites tailles, 2 époques) — vérifie que tout fonctionne
import importlib, shutil, sys
sys.path.insert(0, '.')
import prach_sim, features, experiments, analysis
for m in (prach_sim, features, experiments, analysis): importlib.reload(m)
shutil.rmtree('/content/smoke', ignore_errors=True)
experiments.run_all(dict(n_runs=1, n_train_cell=6, n_val_cell=3, n_test_cell=3, n_rob_cell=2, epochs=2,
                         out_dir='/content/smoke'))
print('Test rapide OK')

In [ ]:
# 4. Campagne complète (environ 3 à 5 h sur GPU T4). Reprise automatique si interrompue.
import time, sys
sys.path.insert(0, '.')
import prach_sim, features, experiments, analysis
t0 = time.time()
experiments.run_all(dict(out_dir=OUT), log=lambda s: print(time.strftime('%H:%M:%S'), s, flush=True))
print(f'Terminé en {(time.time()-t0)/3600:.2f} h')

In [ ]:
# 5. Statistiques, tableaux LaTeX, figures, archive à télécharger
import pandas as pd, json, shutil, os, sys
sys.path.insert(0, '.')
import analysis
S, res = analysis.full_report(OUT)
cols = ['model','n_runs','accuracy_mean','accuracy_std','macro_f1_mean','macro_f1_std','macro_f1_ci95',
        'test_unseen_roots_macro_f1_mean','test_EPA5_macro_f1_mean','fit_s_mean','infer_ms_per_sample_mean','n_params_mean']
display(S[cols].round(4))
print('Friedman (macro-F1) :', res['macro_f1'].get('friedman_chi2'), 'p =', res['macro_f1'].get('friedman_p'))
display(pd.DataFrame(res['macro_f1']['pairwise']).round(4))
shutil.make_archive('/content/results', 'zip', OUT)
shutil.copy('/content/results.zip', os.path.dirname(OUT) + '/results.zip')
from google.colab import files
files.download('/content/results.zip')